In [4]:
pip install easyocr

  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 26.5 MB/s  0:00:00
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ---------------------------------------- 4.1/4.1 MB 24.1 MB/s  0:00:00
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---- ----------------------------------- 13.1/123.0 MB 63.9 MB/s eta 0:00:02
   -------- ------------------------------- 27.5/123.0 MB 66.0 MB/s eta 0:00:02
   ------------- -------------------------- 42.2/123.0 MB 67.2 MB/s eta 0:00:02
   ------------------ --------------------- 57.4/123.0 MB 68.0 MB/s eta 0:00:01
   ----------------------- ---------------- 72.4/123.0 MB 68.6 MB/s eta 0:00:01
   ---------------------------- ----------- 87.6/123.0 MB 69.0 MB/s eta 0:00:01
   -------------------------------- ------ 102.8/123.0 MB 69.3 MB/s eta 0:

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\micha\\anaconda3\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.



In [ ]:
import cv2
import pytesseract
from pytesseract import Output
import numpy as np
import os
import csv
import json
from skimage.metrics import structural_similarity as ssim
from datetime import timedelta

# ---------------- CONFIG ----------------
VIDEO_FILE = "2026-05-13-graduate.mp4"

START_TIME = "00:38:02"
END_TIME   = "01:12:02"

ANNOTATED_DIR = "annotated"
CSV_FILE = "ocr_results.csv"
JSON_FILE = "ocr_results.json"

os.makedirs(ANNOTATED_DIR, exist_ok=True)

# OCR ROI
X0, Y0 = 250, 520
X1, Y1 = 1920, 1080

# OCR config
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
TESS_CFG = "--psm 6 --oem 3 -l eng"

def ts_to_seconds(ts):
    h, m, s = ts.split(":")
    return int(h)*3600 + int(m)*60 + float(s)

def format_timestamp(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:06.3f}"

# ---------------- RESUME LOGIC ----------------
resume_sec = ts_to_seconds(START_TIME)

if os.path.exists(CSV_FILE):
    with open(CSV_FILE, "r", encoding="utf-8") as f:
        rows = list(csv.reader(f))
        if len(rows) > 1:
            last_ts = rows[-1][0]
            resume_sec = ts_to_seconds(last_ts)
            print(f"Found existing CSV file: {CSV_FILE}")
            print(f"Resuming last location of: {last_ts}")
        else:
            print("CSV exists but contains no detections, starting fresh.")
else:
    print("No CSV found. Starting from beginning.")

# ---------------- VIDEO SETUP ----------------
cap = cv2.VideoCapture(VIDEO_FILE)
fps = cap.get(cv2.CAP_PROP_FPS)

cap.set(cv2.CAP_PROP_POS_MSEC, resume_sec * 1000)

last_roi = None
frame_number = int(resume_sec * fps)

# ---------------- JSON STREAMING SETUP ----------------
json_file = open(JSON_FILE, "a", encoding="utf-8")
if os.path.getsize(JSON_FILE) == 0:
    json_file.write("[\n")
    first_json = True
else:
    json_file.seek(0, os.SEEK_END)
    first_json = False

# ---------------- CSV SETUP ----------------
csv_exists = os.path.exists(CSV_FILE)
csv_file = open(CSV_FILE, "a", newline="", encoding="utf-8")
writer = csv.writer(csv_file)

if not csv_exists:
    writer.writerow(["timestamp", "ocr_text", "confidence"])

# ---------------- MAIN LOOP ----------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    current_sec = resume_sec + ((frame_number - int(resume_sec * fps)) / fps)
    if current_sec > ts_to_seconds(END_TIME):
        break

    roi = frame[Y0:Y1, X0:X1]

    changed = True
    if last_roi is not None:
        roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        last_gray = cv2.cvtColor(last_roi, cv2.COLOR_BGR2GRAY)
        roi_gray = cv2.resize(roi_gray, (last_gray.shape[1], last_gray.shape[0]))
        score = ssim(roi_gray, last_gray)
        changed = score < 0.92

    if changed:
        # OCR
        data = pytesseract.image_to_data(roi, config=TESS_CFG, output_type=Output.DICT)
        text = pytesseract.image_to_string(roi, config=TESS_CFG).strip()

        confs = []
        for c in data["conf"]:
            try:
                ci = int(c)
                if ci > 0:
                    confs.append(ci)
            except:
                pass
        avg_conf = sum(confs)/len(confs) if confs else 0.0

        last_roi = roi.copy()

        # Annotated frame
        annotated = frame.copy()
        cv2.rectangle(annotated, (X0, Y0), (X1, Y1), (0,255,0), 3)

        if text:
            y0 = 40
            for line in text.split("\n"):
                cv2.putText(annotated, line, (50, y0),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                            (0,255,0), 2, cv2.LINE_AA)
                y0 += 40

        # Timestamp in upper-right
        ts_str = format_timestamp(current_sec)
        (tw, th), _ = cv2.getTextSize(ts_str, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)
        x = annotated.shape[1] - tw - 50
        y = 50 + th
        cv2.putText(annotated, ts_str, (x, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,255), 3, cv2.LINE_AA)

        # Save annotated frame
        ann_path = os.path.join(ANNOTATED_DIR, f"{frame_number:06d}_annotated.jpg")
        cv2.imwrite(ann_path, annotated)

        # CSV write
        writer.writerow([ts_str, text, f"{avg_conf:.2f}"])
        csv_file.flush()

        # JSON write
        entry = {
            "index": str(frame_number),
            "timestamp": ts_str,
            "label": "frame",
            "ocr_text": text,
            "confidence": f"{avg_conf:.2f}"
        }

        if not first_json:
            json_file.write(",\n")
        first_json = False

        json_file.write(json.dumps(entry, indent=2))
        json_file.flush()
        os.fsync(json_file.fileno())

        # ---------------- TERMINAL STATUS UPDATE ----------------
        print(f"[DETECTION] {ts_str} — \"{text}\"")

    frame_number += 1

# Close JSON array
json_file.write("\n]\n")
json_file.close()
csv_file.close()
cap.release()

print("Done.")


No CSV found. Starting from beginning.
[DETECTION] 00:38:02.000 — ";"
[DETECTION] 00:38:06.538 — "Jiann-Ping Hsu College of Public Health"
[DETECTION] 00:38:11.109 — "Breyonna Brown"
[DETECTION] 00:38:17.449 — "Kayla Diane Howard
Cum Laude"
[DETECTION] 00:38:24.990 — "Laura Esquivel"
[DETECTION] 00:38:31.763 — "Parker College of Business"
[DETECTION] 00:38:34.733 — "Summer Denise Linton"
[DETECTION] 00:38:51.983 — "JeCory Cle’mons Robinson"
[DETECTION] 00:38:57.355 — "Brett Bronistaw Duszak"
[DETECTION] 00:39:06.531 — "Christopher Robert Bunker"
[DETECTION] 00:39:17.475 — "Giselle Gobi Pereira
Summa Cum Laude"
[DETECTION] 00:39:23.248 — "Benjamin Louis Stolba"
[DETECTION] 00:39:28.353 — "Emerson Grant Mitchell
Cum Laude"
[DETECTION] 00:39:35.660 — "Matthew Joseph Ristuccia"
[DETECTION] 00:39:41.333 — "Gloria Adalgela Corniel
Magna Cum Laude"
[DETECTION] 00:39:46.905 — "Jhalil Jarid Smith"
[DETECTION] 00:39:58.249 — "Linea Elizabeth Romano
Magna Cum Laude"
[DETECTION] 00:40:08.793 — "Ad

## For some reason the above did not capture abligail the 2nd time.  

In [1]:
import cv2
import pytesseract
from pytesseract import Output
import numpy as np
import os
import csv
import json
from skimage.metrics import structural_similarity as ssim
from datetime import timedelta

# ---------------- CONFIG ----------------
VIDEO_FILE = "2026-05-13-graduate.mp4"

START_TIME = "00:38:02"
END_TIME   = "01:12:02"

ANNOTATED_DIR = "annotated"
CSV_FILE = "ocr_results.csv"
JSON_FILE = "ocr_results.json"


# Diff Threshold
difference_pct = 0.99
fps_to_check = 3

#changed = score < 0.95
#changed = score < difference_pct

os.makedirs(ANNOTATED_DIR, exist_ok=True)

# OCR ROI
X0, Y0 = 250, 520
X1, Y1 = 1920, 1080


# OCR config
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
TESS_CFG = "--psm 6 --oem 3 -l eng"

def ts_to_seconds(ts):
    h, m, s = ts.split(":")
    return int(h)*3600 + int(m)*60 + float(s)

def format_timestamp(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:06.3f}"

# ---------------- RESUME LOGIC ----------------
resume_sec = ts_to_seconds(START_TIME)

if os.path.exists(CSV_FILE):
    with open(CSV_FILE, "r", encoding="utf-8") as f:
        rows = list(csv.reader(f))
        if len(rows) > 1:
            last_ts = rows[-1][0]
            resume_sec = ts_to_seconds(last_ts)
            print(f"Found existing CSV file: {CSV_FILE}")
            print(f"Resuming last location of: {last_ts}")
        else:
            print("CSV exists but contains no detections, starting fresh.")
else:
    print("No CSV found. Starting from beginning.")

# ---------------- VIDEO SETUP ----------------
cap = cv2.VideoCapture(VIDEO_FILE)
fps = cap.get(cv2.CAP_PROP_FPS)

# Calculate how many frames equal 1/3rd of a second (fallback to 1 if fps is missing)
frames_per_check = max(1, int(fps / fps_to_check)) ## NOTE: Here is where we would set the frames per second to check

cap.set(cv2.CAP_PROP_POS_MSEC, resume_sec * 1000)

last_roi = None
frame_number = int(resume_sec * fps)

# ---------------- JSON STREAMING SETUP ----------------
json_file = open(JSON_FILE, "a", encoding="utf-8")
if os.path.getsize(JSON_FILE) == 0:
    json_file.write("[\n")
    first_json = True
else:
    json_file.seek(0, os.SEEK_END)
    first_json = False

# ---------------- CSV SETUP ----------------
csv_exists = os.path.exists(CSV_FILE)
csv_file = open(CSV_FILE, "a", newline="", encoding="utf-8")
writer = csv.writer(csv_file)

if not csv_exists:
    writer.writerow(["timestamp", "ocr_text", "confidence"])

# ---------------- MAIN LOOP ----------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Skip frames to force processing exactly 3 times a second
    if frame_number % frames_per_check != 0:
        frame_number += 1
        continue

    current_sec = resume_sec + ((frame_number - int(resume_sec * fps)) / fps)
    if current_sec > ts_to_seconds(END_TIME):
        break

    roi = frame[Y0:Y1, X0:X1]

    changed = True
    if last_roi is not None:
        roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        last_gray = cv2.cvtColor(last_roi, cv2.COLOR_BGR2GRAY)
        roi_gray = cv2.resize(roi_gray, (last_gray.shape[1], last_gray.shape[0]))
        score = ssim(roi_gray, last_gray)
        
        # Bumped threshold from 0.92 to 0.95 to be more sensitive to name changes
        changed = score < difference_pct

    if changed:
        # OCR
        data = pytesseract.image_to_data(roi, config=TESS_CFG, output_type=Output.DICT)
        text = pytesseract.image_to_string(roi, config=TESS_CFG).strip()

        confs = []
        for c in data["conf"]:
            try:
                ci = int(c)
                if ci > 0:
                    confs.append(ci)
            except:
                pass
        avg_conf = sum(confs)/len(confs) if confs else 0.0

        last_roi = roi.copy()

        # Annotated frame
        annotated = frame.copy()
        cv2.rectangle(annotated, (X0, Y0), (X1, Y1), (0,255,0), 3)

        if text:
            y0 = 40
            for line in text.split("\n"):
                cv2.putText(annotated, line, (50, y0),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                            (0,255,0), 2, cv2.LINE_AA)
                y0 += 40

        # Timestamp in upper-right
        ts_str = format_timestamp(current_sec)
        (tw, th), _ = cv2.getTextSize(ts_str, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)
        x = annotated.shape[1] - tw - 50
        y = 50 + th
        cv2.putText(annotated, ts_str, (x, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,255), 3, cv2.LINE_AA)

        # Save annotated frame
        ann_path = os.path.join(ANNOTATED_DIR, f"{frame_number:06d}_annotated.jpg")
        cv2.imwrite(ann_path, annotated)

        # CSV write
        writer.writerow([ts_str, text, f"{avg_conf:.2f}"])
        csv_file.flush()

        # JSON write
        entry = {
            "index": str(frame_number),
            "timestamp": ts_str,
            "label": "frame",
            "ocr_text": text,
            "confidence": f"{avg_conf:.2f}"
        }

        if not first_json:
            json_file.write(",\n")
        first_json = False

        json_file.write(json.dumps(entry, indent=2))
        json_file.flush()
        os.fsync(json_file.fileno())

        # ---------------- TERMINAL STATUS UPDATE ----------------
        print(f"[DETECTION] {ts_str} — \"{text}\"")

    frame_number += 1

# Close JSON array
json_file.write("\n]\n")
json_file.close()
csv_file.close()
cap.release()

print("Done.")

Found existing CSV file: ocr_results.csv
Resuming last location of: 00:44:45.003
[DETECTION] 00:44:45.003 — "Chukwukelum Williams Okoye"
[DETECTION] 00:44:51.009 — "Javierre Gabriel Williams"
[DETECTION] 00:44:58.517 — "Bao Quoc Pham
Magna Cum Laude"
[DETECTION] 00:45:03.321 — "Windy Nguyen Pham
Cum Laude"
[DETECTION] 00:45:08.727 — "Maisie Xuan Mai Tran"
[DETECTION] 00:45:13.832 — "Gawain Nathaniel Mendez"
[DETECTION] 00:45:19.237 — "Michael Leonard Woodcock
Magna Cum Laude"
[DETECTION] 00:45:24.342 — "Kaylee B. Mailler
Magna Cum Laude"
[DETECTION] 00:45:30.048 — "Brent William Shepherd"
[DETECTION] 00:45:35.453 — "Kadija Seisay"
[DETECTION] 00:45:41.159 — "Lesly Melinda Zetino-Alas"
[DETECTION] 00:45:48.667 — "College of Science and Mathematics"
[DETECTION] 00:45:51.970 — "Sophia Castillo-Gomez"
[DETECTION] 00:45:56.775 — "Katie Nguyen
Magna Cum Laude"
[DETECTION] 00:46:03.081 — "Kayla Allyse Barnes
Magna Cum Laude"
[DETECTION] 00:46:08.186 — "Van Evans
Cum Laude"
[DETECTION] 00:46:1